## Distiller SimSwap 512 pour remplacer son générateur

Exécuter `dataset_prep.ipynb` avant ce notebook. L'élève reçoit exactement
`(X0, E1)` : RGB float32 NCHW en `[0, 1]`, crop ffhq 512 et embedding **SimSwap**
L2 de dimension 512. Il renvoie une image RGB de même forme en `[0, 1]`.
`X1` est le portrait donneur permettant de calculer E1, pas une deuxième entrée image
du générateur. Le backend continuera d'assurer détection, alignement, ArcFace et blending.

L'élève conserve l'architecture officielle, avec 8 blocs AdaIN au lieu de 9.
Les couches correspondantes sont initialisées avec les poids du maître ; seul son
dernier bloc résiduel est retiré. La réduction de paramètres est mesurée plus bas.
Le discriminateur sert uniquement à l'entraînement et n'est pas exporté pour l'inférence.

Les cibles sont les sorties Y pré-calculées, sans BiSeNet. La perte de reconstruction
compare l'élève à Y, jamais à X0. La validation suit des X0 absents de l'entraînement ;
elle ne mesure pas encore la généralisation à de nouveaux donneurs. La convergence
et la qualité doivent être vérifiées sur les courbes et les images, pas supposées.

In [ ]:
# --- 1. IMPORTS ET REPRODUCTIBILITÉ ---
import gc
import json
import random
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from src.utils.config import Settings
from src.utils.image_utils import load_image
from src.utils.model_manager import sha256
from src.backends.simswap_backend import SimSwapBackend, add_official_source

logging.getLogger().setLevel(logging.ERROR)
settings = Settings(device="cuda" if torch.cuda.is_available() else "cpu",
                    crop_size=512, segmentation=False)
device = settings.device
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
add_official_source(settings.simswap)
from models.fs_networks_512 import Generator_Adain_Upsample
print(f"L'entraînement s'effectuera sur : {device.upper()}")

In [ ]:
# --- 2. ARCHITECTURE DU GÉNÉRATEUR ALLÉGÉ (L'ÉLÈVE) ---
class CustomLiteGenerator(Generator_Adain_Upsample):
    def __init__(self, style_dim=512, num_blocks=8):
        super().__init__(input_nc=3, output_nc=3, latent_size=style_dim,
                         n_blocks=num_blocks, deep=False)
        self.style_dim = style_dim
        self.num_blocks = num_blocks
    # Le forward officiel conserve (tanh(x) + 1) / 2 : sortie RGB [0, 1].

# --- ARCHITECTURE DU DISCRIMINATEUR CONDITIONNÉ PAR X0 ---
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.Conv2d(256, 1, kernel_size=4, padding=1)
        )

    def forward(self, x0, image):
        # Discrimination à 256 pixels pour limiter la mémoire à l'entraînement.
        return self.model(F.avg_pool2d(torch.cat((x0, image), dim=1), 2))

In [ ]:
# --- 3. CHARGEMENT DES TRIPLETS X0, X1, E1, Y ---
dataset_dir = Path("processed_dataset_512/distillation")
manifest_path = dataset_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
contract = manifest["configuration"]
if (contract["version"] != 1 or contract["crop_size"] != 512
        or contract["alignment"] != "ffhq" or contract["color"] != "RGB"
        or contract["image_range"] != [0, 1]):
    raise ValueError("Le dataset ne respecte pas le contrat SimSwap 512")
if contract["preprocessing_hashes"]["src/backends/simswap_backend.py"] != sha256(
        settings.root / "src/backends/simswap_backend.py"):
    raise ValueError("Le backend de référence a changé depuis la préparation du dataset")

class FaceSwapDataset(Dataset):
    def __init__(self, root, records):
        self.root = Path(root)
        self.records = records
        if not self.records:
            raise ValueError("Split vide : relancer la préparation du dataset")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        def image_tensor(key):
            rgb = load_image(self.root / record[key])[..., ::-1]
            return torch.from_numpy(rgb.transpose(2, 0, 1).copy()).float() / 255
        x0, x1 = image_tensor("X0"), image_tensor("X1")
        identity = torch.from_numpy(np.load(self.root / record["E1"], allow_pickle=False).copy())
        y = torch.from_numpy(np.load(self.root / record["Y"], allow_pickle=False).copy())
        if any(image.shape != (3, 512, 512) for image in (x0, x1, y)):
            raise ValueError(f"Dimensions invalides : {record['key']}")
        if identity.shape != (512,) or not torch.isfinite(identity).all() or not torch.isclose(
                identity.norm(), torch.tensor(1.), atol=1e-4):
            raise ValueError(f"Identité SimSwap invalide : {record['key']}")
        if y.dtype != torch.float32 or not torch.isfinite(y).all() or y.min() < 0 or y.max() > 1:
            raise ValueError(f"Cible Y invalide : {record['key']}")
        return x0, x1, identity, y

train_records = [row for row in manifest["records"] if row["split"] == "train"]
val_records = [row for row in manifest["records"] if row["split"] == "validation"]
if {row["source_id"] for row in train_records} & {row["source_id"] for row in val_records}:
    raise ValueError("Une image X0 apparaît dans les deux splits")
train_dataset = FaceSwapDataset(dataset_dir, train_records)
val_dataset = FaceSwapDataset(dataset_dir, val_records)
batch_size = 2  # Réduire à 1 si la VRAM est insuffisante
dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0,
                        pin_memory=device == "cuda", generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
print(f"Dataset prêt : {len(train_dataset)} exemples train, {len(val_dataset)} validation.")

In [ ]:
# --- 4. INITIALISATION DE L'ÉLÈVE À PARTIR DU MAÎTRE OFFICIEL ---
teacher = SimSwapBackend(settings)  # Chargement strict des poids actifs via le backend existant
from src.utils.model_manager import ModelManager
manager = ModelManager(settings)
for key in ("simswap512", "arcface"):
    if sha256(manager.require(key)) != contract["model_hashes"][key]:
        raise ValueError(f"Le checkpoint {key} diffère de celui qui a produit le dataset")

student_model = CustomLiteGenerator(num_blocks=8)
expected_keys = student_model.state_dict()
inherited = {key: value for key, value in teacher.generator.state_dict().items()
             if key in expected_keys}
student_model.load_state_dict(inherited, strict=True)
student_model = student_model.to(device)
teacher_params = sum(p.numel() for p in teacher.generator.parameters())
student_params = sum(p.numel() for p in student_model.parameters())
assert student_params < teacher_params
print(f"Maître : {teacher_params:,} paramètres ; élève : {student_params:,}")
print(f"Réduction du générateur : {100 * (1 - student_params / teacher_params):.2f} %")

# Garder seulement ArcFace pour la perte d'identité, avec gradient vers l'image générée.
arcface_model = teacher.arcface.eval().requires_grad_(False)
arc_mean, arc_std = teacher.mean, teacher.std
del expected_keys, inherited, teacher
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
discriminator = PatchDiscriminator().to(device)

In [ ]:
# --- 5. PERTES ET OPTIMISEURS ---
criterion_GAN = nn.MSELoss()
criterion_pixel = nn.L1Loss()

def distillation_loss(fake, y):
    # Superviser Y à plusieurs échelles, sans pénaliser le changement d'identité vers X1.
    loss = criterion_pixel(fake, y)
    for scale in (2, 4):
        loss = loss + criterion_pixel(F.avg_pool2d(fake, scale), F.avg_pool2d(y, scale))
    return loss / 3

def gradient_loss(fake, y):
    horizontal = criterion_pixel(fake[:, :, :, 1:] - fake[:, :, :, :-1],
                                 y[:, :, :, 1:] - y[:, :, :, :-1])
    vertical = criterion_pixel(fake[:, :, 1:, :] - fake[:, :, :-1, :],
                               y[:, :, 1:, :] - y[:, :, :-1, :])
    return (horizontal + vertical) / 2

def identity_loss(fake, identity):
    # Même ArcFace et même prétraitement que pour E1 dans l'application.
    arc_input = F.interpolate((fake - arc_mean) / arc_std, (112, 112), mode="nearest")
    predicted = F.normalize(arcface_model(arc_input), p=2, dim=1)
    return (1 - (predicted * identity).sum(dim=1)).mean()

optimizer_G = optim.Adam(student_model.parameters(), lr=0.0001, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.999))
scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(optimizer_G, factor=0.5, patience=3)
print("Pertes : distillation vers Y, contours, identité X1 et adversarial progressif.")

In [ ]:
# --- 6. ENTRAÎNEMENT ET VALIDATION ---
num_epochs = 50
warmup_epochs = 5  # Stabiliser la distillation avant d'activer le discriminateur
patience_max = 8
best_val_loss = float("inf")
patience_counter = 0
history = []
checkpoint_dir = Path("outputs/gan_maison")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Lancement de l'entraînement...")
for epoch in range(num_epochs):
    student_model.train()
    discriminator.train()
    totals = dict(G=0., D=0., distill=0., identity=0.)
    seen = 0
    adversarial_weight = 0.05 * min(1., max(0., (epoch - warmup_epochs + 1) / 5))

    for batch_idx, (x0, x1, identity, y) in enumerate(dataloader):
        x0, identity, y = x0.to(device), identity.to(device), y.to(device)
        optimizer_G.zero_grad()
        fake = student_model(x0, identity)

        # PHASE 1 : LE DISCRIMINATEUR COMPARE Y ET LA SORTIE DE L'ÉLÈVE
        loss_D = torch.zeros((), device=device)
        if adversarial_weight > 0:
            discriminator.requires_grad_(True)
            optimizer_D.zero_grad()
            pred_real = discriminator(x0, y)
            pred_fake = discriminator(x0, fake.detach())
            loss_D = (criterion_GAN(pred_real, torch.ones_like(pred_real))
                      + criterion_GAN(pred_fake, torch.zeros_like(pred_fake))) / 2
            if not torch.isfinite(loss_D):
                raise FloatingPointError("Perte discriminateur non finie")
            loss_D.backward()
            optimizer_D.step()

        # PHASE 2 : DISTILLATION ET IDENTITÉ DU DONNEUR
        discriminator.requires_grad_(False)
        loss_distill = distillation_loss(fake, y)
        loss_id = identity_loss(fake, identity)
        loss_G = 10 * loss_distill + gradient_loss(fake, y) + loss_id
        if adversarial_weight > 0:
            pred_fake_for_G = discriminator(x0, fake)
            loss_G = loss_G + adversarial_weight * criterion_GAN(
                pred_fake_for_G, torch.ones_like(pred_fake_for_G))
        if not torch.isfinite(loss_G):
            raise FloatingPointError("Perte générateur non finie")
        loss_G.backward()
        nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=5., error_if_nonfinite=True)
        optimizer_G.step()

        count = x0.shape[0]
        seen += count
        for name, value in (("G", loss_G), ("D", loss_D),
                            ("distill", loss_distill), ("identity", loss_id)):
            totals[name] += value.detach().item() * count
        if batch_idx % 25 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs} | Batch {batch_idx}/{len(dataloader)} "
                  f"| G={loss_G.item():.4f} | D={loss_D.item():.4f}")

    # VALIDATION SUR DES X0 DISTINCTS, SANS GRADIENT
    student_model.eval()
    val_total, val_seen = 0., 0
    with torch.inference_mode():
        for x0, x1, identity, y in val_loader:
            x0, identity, y = x0.to(device), identity.to(device), y.to(device)
            prediction = student_model(x0, identity)
            val_total += distillation_loss(prediction, y).item() * x0.shape[0]
            val_seen += x0.shape[0]
    val_loss = val_total / val_seen
    if not np.isfinite(val_loss):
        raise FloatingPointError("Perte validation non finie")
    scheduler_G.step(val_loss)
    row = dict(epoch=epoch + 1, **{name: value / seen for name, value in totals.items()},
               validation=val_loss, lr=optimizer_G.param_groups[0]["lr"])
    history.append(row)
    (checkpoint_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    print(f"Epoch {epoch + 1} | distillation train={row['distill']:.5f} | validation={val_loss:.5f}")

    # EARLY STOPPING SUR LA MOYENNE DE VALIDATION, PAS SUR LE DERNIER BATCH TRAIN
    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        torch.save(student_model.state_dict(), checkpoint_dir / "lite_generator_best.pth")
        model_config = dict(architecture="CustomLiteGenerator / SimSwap512", style_dim=512,
                            num_blocks=8, deep=False, crop_size=512, alignment="ffhq",
                            color="RGB", image_range=[0, 1], identity="SimSwap ArcFace L2",
                            teacher_params=teacher_params, student_params=student_params,
                            epoch=epoch + 1, validation=val_loss,
                            model_hashes=contract["model_hashes"],
                            manifest_sha256=sha256(manifest_path))
        (checkpoint_dir / "lite_generator_best.json").write_text(
            json.dumps(model_config, indent=2), encoding="utf-8")
        print("Meilleur élève enregistré !")
    else:
        patience_counter += 1
        if patience_counter >= patience_max:
            print("Arrêt anticipé : la validation ne progresse plus.")
            break
print("Entraînement terminé !")

In [ ]:
# --- 7. COURBES ET CONTRÔLE DU MEILLEUR ÉLÈVE ---
student_model.load_state_dict(torch.load(checkpoint_dir / "lite_generator_best.pth",
                                        map_location=device, weights_only=True), strict=True)
student_model.eval()
plt.figure(figsize=(8, 4))
plt.plot([row["epoch"] for row in history], [row["distill"] for row in history], label="Train vers Y")
plt.plot([row["epoch"] for row in history], [row["validation"] for row in history], label="Validation vers Y")
plt.xlabel("Epoch")
plt.ylabel("L1 multi-échelle")
plt.legend()
plt.show()

x0, x1, identity, y = next(iter(val_loader))
with torch.inference_mode():
    prediction = student_model(x0.to(device), identity.to(device)).cpu()
assert prediction.shape == x0.shape and torch.isfinite(prediction).all()
assert prediction.min() >= 0 and prediction.max() <= 1
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, title, image in zip(axes, ("X0", "X1 donneur", "Y maître", "Élève"),
                            (x0[0], x1[0], y[0], prediction[0])):
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Contrat vérifié : générateur (RGB [0, 1], identité SimSwap L2) -> RGB [0, 1].")